# Black-Box Optimisation BBO




## Introduction

A black-box optimization project involves finding the input values that maximize an unknown function when you can only observe its outputs, not its internal structure. This means to intelligently explore eight separate functions—each with different input dimensions bounded between 0 and 1—to efficiently discover the combinations that yield the highest possible output.

In [99]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from scipy.optimize import minimize
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

## Provided initial data and weekly evaluations

In [100]:
# Load the initial data provided
f1_input_data = np.load('./../initial_data/function_1/initial_inputs.npy')
f1_output_data = np.load('./../initial_data/function_1/initial_outputs.npy')

# Weekly submissions
f1_input_data = np.concatenate(
    (f1_input_data, [[0.223696, 0.166025], # Week 1
                     [0.971666, 0.994621], # Week 2 
                     [0.580800, 0.403103], # Week 3
                     [0.530137, 0.504108], # Week 4
                     [0.284100, 0.754200], # Week 5
                     [0.879672, 0.190177], # Week 6
                     [0.047162, 0.569522], # Week 7
                     [0.608963, 0.555379], # Week 8
                     [0.568947, 0.768862], # Week 9
                     [0.388166, 0.232975], # Week 10
                     [0.881147, 0.795664], # Week 11
                     [0.339981, 0.323487], # Week 12
                     [0.390337, 0.301715]]), # Week 13
    axis=0
)
# Results (weekly evaluation outputs)
f1_output_data = np.append(f1_output_data, 1.2121776220527492e-74) # GP + UCB (iterations = 10) kappa = 8.0
f1_output_data = np.append(f1_output_data, -4.515332508840043e-176) # GP + UCB (iterations = 10) kappa = 20.0
f1_output_data = np.append(f1_output_data, 9.827384409509805e-19) # GP + UCB (iterations = 10) kappa = 12.0  - kappa 10 ????
f1_output_data = np.append(f1_output_data, 3.9242133027962733e-14) # GP + UCB (iterations = 10) kappa = 2.0 - kappa 9 ???
f1_output_data = np.append(f1_output_data, 4.690604757883212e-91) # Neural Networks + UCB (iterations = 10) kappa 2.0
f1_output_data = np.append(f1_output_data, -1.8323318855977174e-178) # GP + EI (iterations = 10) kappa = 0.05
f1_output_data = np.append(f1_output_data, -1.80712027198355e-113) # GP + EI (iterations = 15) kappa = 5.0
f1_output_data = np.append(f1_output_data, -0.0001431460817369449) # GP + EI (iterations = 15) kappa = 10.0
f1_output_data = np.append(f1_output_data, -8.059845136259036e-17) # GP + UCB (iterations = 10) kappa = 9.5
f1_output_data = np.append(f1_output_data, 2.559306584393459e-27) # GP + UCB (iterations = 10) kappa = 5.0
f1_output_data = np.append(f1_output_data, -2.55746372896521e-64) # GP + UCB (iterations = 10) kappa = 6.0
f1_output_data = np.append(f1_output_data, 3.688998813902206e-12) # GP + UCB (iterations = 10) kappa = 0.2
f1_output_data = np.append(f1_output_data, 5.714209335193049e-13) # GP + UCB (iterations = 10) kappa = 0.1

In [86]:
# Load the initial data provided
f2_input_data = np.load('./../initial_data/function_2/initial_inputs.npy')
f2_output_data = np.load('./../initial_data/function_2/initial_outputs.npy')

# Weekly submissions
f2_input_data = np.concatenate(
    (f2_input_data, [[0.125906, 0.788607], # Week 1
                     [0.985180, 0.251085], # Week 2
                     [0.441830, 0.633407], # Week 3
                     [0.679819, 0.266237], # Week 4
                     [0.657000, 0.212300], # Week 5
                     [0.790743, 0.938552], # Week 6
                     [0.519940, 0.509866], # Week 7
                     [0.703298, 0.466477], # Week 8
                     [0.534528, 0.278183], # Week 9
                     [0.536392, 0.918154], # Week 10
                     [0.011421, 0.198983], # Week 11
                     [0.591362, 0.428469], # Week 12
                     [0.510208, 0.529779]]), # Week 13
    axis=0
)
# Results (weekly evaluation outputs)
f2_output_data = np.append(f2_output_data, -0.1739726394059063) # GP + UCB (iterations = 10) kappa = 8.0
f2_output_data = np.append(f2_output_data, -0.01550891697481739) # GP + UCB (iterations = 10) kappa = 20.0
f2_output_data = np.append(f2_output_data, 0.2984716850359006) # GP + UCB (iterations = 10) kappa = 12.0
f2_output_data = np.append(f2_output_data, 0.3520637568002779) # GP + UCB (iterations = 10) kappa = 2.0
f2_output_data = np.append(f2_output_data, 0.36843190917654645) # Neural Networks + UCB (iterations = 10) kappa 2.0
f2_output_data = np.append(f2_output_data, 0.035725127837916065) # GP + EI (iterations = 10) kappa = 0.05
f2_output_data = np.append(f2_output_data, 0.6144819505511563) # GP + EI (iterations = 15) kappa = 5.0
f2_output_data = np.append(f2_output_data, 0.5107611479314056) # GP + EI (iterations = 15) kappa = 10.0
f2_output_data = np.append(f2_output_data, 0.35073554356687237) # GP + EI (iterations = 15) kappa = 4.5
f2_output_data = np.append(f2_output_data, 0.1266592974188294) # GP + EI (iterations = 10) kappa = 3.0
f2_output_data = np.append(f2_output_data, -0.015050624482960325) # GP + EI (iterations = 10) kappa = 4.0
f2_output_data = np.append(f2_output_data, 0.10868422130010974) # GP + EI (iterations = 10) kappa = 0.2  ?? instead of UCB
f2_output_data = np.append(f2_output_data, 0.7770004599039609) # GP + UCB (iterations = 10) kappa = 0.1

In [87]:
# Load the initial data provided
f3_input_data = np.load('./../initial_data/function_3/initial_inputs.npy')
f3_output_data = np.load('./../initial_data/function_3/initial_outputs.npy')

# Weekly submissions
f3_input_data = np.concatenate(
    (f3_input_data, [[0.370443, 0.100000, 0.100000], # Week 1
                     [0.100000, 0.900000, 0.900000], # Week 2
                     [0.559175, 0.100000, 0.618789], # Week 3
                     [0.418978, 0.452988, 0.516531], # Week 4
                     [0.907500, 0.234800, 0.979900], # Week 5
                     [0.733217, 0.902473, 0.900000], # Week 6
                     [0.056745, 0.644609, 0.812067], # Week 7
                     [0.401965, 0.444137, 0.317518], # Week 8
                     [0.460388, 0.697471, 0.592882], # Week 9
                     [0.710270, 0.560110, 0.653967], # Week 10
                     [0.303306, 0.000000, 0.411004], # Week 11
                     [0.801517, 0.472335, 0.709245], # Week 12
                     [0.437361, 0.461248, 0.586587]]), # Week 13
    axis=0
)
# Results (weekly evaluation outputs)
f3_output_data = np.append(f3_output_data, -0.117375859160532) # GP + UCB (iterations = 10) kappa = 8.0
f3_output_data = np.append(f3_output_data, -0.12336864739293958) # GP + UCB (iterations = 10) kappa = 20.0
f3_output_data = np.append(f3_output_data, -0.11919639248781014) # GP + UCB (iterations = 10) kappa = 2.0 ??? kappa = 2.0
f3_output_data = np.append(f3_output_data, -0.015678917994671488) # GP + UCB (iterations = 10) kappa = 1.0  ??? kappa=1.0
f3_output_data = np.append(f3_output_data, -0.3698004461896308) # Neural Networks + UCB (iterations = 10) kappa 2.0
f3_output_data = np.append(f3_output_data, -0.09232158961187506) # GP + EI (iterations = 10) kappa = 0.05
f3_output_data = np.append(f3_output_data, -0.06693057594200055) # GP + EI (iterations = 15) kappa = 5.0
f3_output_data = np.append(f3_output_data, -0.06099866510741818) # GP + EI (iterations = 15) kappa = 10.0
f3_output_data = np.append(f3_output_data, -0.04791215400572666) # GP + UCB (iterations = 15) kappa = 0.5
f3_output_data = np.append(f3_output_data, -0.1134040954820501) # GP + UCB (iterations = 10) kappa = 1.0
f3_output_data = np.append(f3_output_data, -0.0740441101108158) # GP + UCB (iterations = 10) kappa = 2.0
f3_output_data = np.append(f3_output_data, -0.11724410952717225) # GP + EI (iterations = 10) kappa = 0.2
f3_output_data = np.append(f3_output_data, -0.041054575325136906) # GP + UCB + PCA (iterations = 10) kappa = 0.1

In [88]:
# Load the initial data provided
f4_input_data = np.load('./../initial_data/function_4/initial_inputs.npy')
f4_output_data = np.load('./../initial_data/function_4/initial_outputs.npy')

# Weekly submissions
f4_input_data = np.concatenate(
    (f4_input_data, [[0.900000, 0.391547, 0.100000, 0.100000], # Week 1
                     [0.100000, 0.900000, 0.900000, 0.900000], # Week 2
                     [0.405673, 0.390640, 0.346566, 0.428138], # Week 3
                     [0.407626, 0.408744, 0.294125, 0.437783], # Week 4
                     [0.342600, 0.558700, 0.611600, 0.502200], # Week 5
                     [0.685479, 0.753190, 0.168306, 0.737342], # Week 6
                     [0.577678, 0.500100, 0.677226, 0.175109], # Week 7
                     [0.172470, 0.073209, 0.372759, 0.989174], # Week 8
                     [0.424494, 0.339512, 0.445235, 0.454909], # Week 9
                     [0.679848, 0.950000, 0.950000, 0.950000], # Week 10
                     [0.332322, 0.568071, 0.035973, 0.674249], # Week 11
                     [0.588167, 0.570162, 0.563914, 0.107006], # Week 12
                     [0.587046, 0.596437, 0.489224, 0.572515]]), # Week 13
    axis=0
)
# Results (weekly evaluation outputs)
f4_output_data = np.append(f4_output_data, -19.874502519770903) # GP + UCB (iterations = 10) kappa = 8.0
f4_output_data = np.append(f4_output_data, -34.10576750538245) # GP + UCB (iterations = 10) kappa = 20.0
f4_output_data = np.append(f4_output_data, 0.46654522988047065) # GP + UCB (iterations = 10) kappa = 2.0 ???
f4_output_data = np.append(f4_output_data, -0.978378387130991) # GP + UCB (iterations = 10) kappa = 1.0  ???
f4_output_data = np.append(f4_output_data, -6.075122732992654) # Neural Networks + UCB (iterations = 10) kappa 2.0
f4_output_data = np.append(f4_output_data, -18.37458887015637) # GP + EI (iterations = 10) kappa = 0.05
f4_output_data = np.append(f4_output_data, -10.314157202143814) # GP + EI (iterations = 15) kappa = 5.0
f4_output_data = np.append(f4_output_data, -21.88994067659473) # GP + EI (iterations = 15) kappa = 10.0
f4_output_data = np.append(f4_output_data, -0.3092559607133718) # GP + UCB (iterations = 15) kappa = 0.5
f4_output_data = np.append(f4_output_data, -40.031478179592405) # GP + UCB (iterations = 10) kappa = 4.0
f4_output_data = np.append(f4_output_data, -13.22666268459103) # GP + EI (iterations = 10) kappa = 4.0
f4_output_data = np.append(f4_output_data, -11.466794676721602) # GP + EI (iterations = 10) kappa = 0.2
f4_output_data = np.append(f4_output_data, -8.478525354989117) # GP + UCB + PCA (iterations = 10) kappa = 0.1

In [89]:
# Load the initial data provided
f5_input_data = np.load('./../initial_data/function_5/initial_inputs.npy')
f5_output_data = np.load('./../initial_data/function_5/initial_outputs.npy')

# Weekly submissions
f5_input_data = np.concatenate(
    (f5_input_data, [[0.261146, 0.837257, 0.855685, 0.889760], # Week 1
                     [0.179332, 0.906546, 0.867288, 0.935252], # Week 2
                     [0.430812, 0.766167, 0.350393, 0.947247], # Week 3
                     [0.526768, 0.806704, 0.939242, 0.609982], # Week 4
                     [0.859900, 0.820800, 0.147800, 0.771800], # Week 5
                     [0.988643, 0.690716, 0.985391, 0.856409], # Week 6
                     [0.723047, 0.530092, 0.811697, 0.798753], # Week 7
                     [0.941048, 0.524466, 0.372639, 0.904600], # Week 8
                     [0.987432, 0.690041, 0.984631, 0.856124], # Week 9
                     [0.938284, 0.520819, 0.379362, 0.902919], # Week 10
                     [0.445315, 0.757693, 0.353784, 0.930553], # Week 11
                     [0.942718, 0.526670, 0.368575, 0.905615], # Week 12
                    [0.611960, 0.647068, 0.529179, 0.763135]]), # Week 13
    axis=0
)
# Results (weekly evaluation outputs)
f5_output_data = np.append(f5_output_data, 1003.0955306052545) # GP + UCB (iterations = 10) kappa = 8.0
f5_output_data = np.append(f5_output_data, 1661.4926023721316) # GP + UCB (iterations = 10) kappa = 20.0
f5_output_data = np.append(f5_output_data, 377.4129493399731) # GP + UCB (iterations = 10) kappa = 2.0 ??
f5_output_data = np.append(f5_output_data, 642.9609629945041) # GP + UCB (iterations = 10) kappa = 1.0 ???
f5_output_data = np.append(f5_output_data, 561.8354085559349) # Neural Networks + UCB (iterations = 10) kappa 2.0
f5_output_data = np.append(f5_output_data, 3378.2312831787153) # GP + EI (iterations = 10) kappa = 0.05
f5_output_data = np.append(f5_output_data, 384.0959683165886) # GP + EI (iterations = 15) kappa = 5.0
f5_output_data = np.append(f5_output_data, 821.4588134967668) # GP + EI (iterations = 15) kappa = 10.0
f5_output_data = np.append(f5_output_data, 3350.4545647667915) # GP + UCB (iterations = 15) kappa = 0.5
f5_output_data = np.append(f5_output_data, 799.5449758520845) # GP + UCB (iterations = 10) kappa = 4.0
f5_output_data = np.append(f5_output_data, 314.88713830765084) # GP + UCB (iterations = 10) kappa = 5.0
f5_output_data = np.append(f5_output_data, 835.0079036275938) # GP + UCB (iterations = 10) kappa = 0.2 
f5_output_data = np.append(f5_output_data, 55.941976575026146) # GP + UCB + PCA (iterations = 10) kappa = 0.1

In [90]:
# Load the initial data provided
f6_input_data = np.load('./../initial_data/function_6/initial_inputs.npy')
f6_output_data = np.load('./../initial_data/function_6/initial_outputs.npy')

# Weekly submissions
f6_input_data = np.concatenate(
    (f6_input_data, [[0.100000, 0.100000, 0.100000, 0.100000, 0.100000], # Week 1 
                     [0.900000, 0.900000, 0.100000, 0.100000, 0.900000], # Week 2
                     [0.100000, 0.100000, 0.477694, 0.900000, 0.100000], # Week 3
                     [0.711119, 0.100000, 0.900000, 0.100000, 0.900000], # Week 4
                     [0.120700, 0.480500, 0.335700, 0.771900, 0.529300], # Week 5
                     [0.445702, 0.249091, 0.570676, 0.789337, 0.143917], # Week 6
                     [0.756618, 0.789573, 0.700148, 0.788363, 0.930751], # Week 7
                     [0.224372, 0.323632, 0.710213, 0.752246, 0.958774], # Week 8
                     [0.432407, 0.055250, 0.765700, 0.598069, 0.906864], # Week 9
                     [0.873488, 0.640637, 0.273703, 0.456593, 0.423597], # Week 10
                     [0.663636, 0.734306, 0.985001, 0.291751, 0.164318], # Week 11
                     [0.431880, 0.230272, 0.644232, 0.969621, 0.166391], # Week 12
                     [0.486650, 0.422857, 0.704530, 0.789187, 0.334515]]), # Week 13
    axis=0
)
# Results (weekly evaluation outputs)
f6_output_data = np.append(f6_output_data, -1.772464173686114) # GP + UCB (iterations = 10) kappa = 8.0
f6_output_data = np.append(f6_output_data, -3.0710848428181494) # GP + UCB (iterations = 10) kappa = 20.0
f6_output_data = np.append(f6_output_data, -0.8420881702572393) # GP + UCB (iterations = 10) kappa = 4.0 ?????
f6_output_data = np.append(f6_output_data, -2.25609789259345) # GP + UCB (iterations = 10) kappa = 2.0 ?????
f6_output_data = np.append(f6_output_data, -1.2030646453198108) # Neural Networks + UCB (iterations = 10) kappa 2.0
f6_output_data = np.append(f6_output_data, -0.27914755272875114) # GP + EI (iterations = 10) kappa = 0.05
f6_output_data = np.append(f6_output_data, -1.6262062830456927) # GP + EI (iterations = 15) kappa = 5.0
f6_output_data = np.append(f6_output_data, -1.2556062668293657) # GP + EI (iterations = 15) kappa = 10.0
f6_output_data = np.append(f6_output_data, -1.4916821089090682) # GP + EI (iterations = 15) kappa = 0.5
f6_output_data = np.append(f6_output_data, -1.8078362346120846) # GP + EI (iterations = 10) kappa = 3.0
f6_output_data = np.append(f6_output_data, -1.4564902307553127) # GP + EI (iterations = 10) kappa = 4.0
f6_output_data = np.append(f6_output_data, -0.4295074199665059) # GP + UCB (iterations = 10) kappa = 0.2
f6_output_data = np.append(f6_output_data, -0.36033278793561724) # GP + UCB + PCA (iterations = 10) kappa = 0.1

In [91]:
# Load the initial data provided
f7_input_data = np.load('./../initial_data/function_7/initial_inputs.npy')
f7_output_data = np.load('./../initial_data/function_7/initial_outputs.npy')

# Weekly submissions
f7_input_data = np.concatenate(
    (f7_input_data, [[0.158904, 0.131800, 0.383454, 0.110947, 0.292992, 0.900000], # Week 1
                     [0.900000, 0.900000, 0.900000, 0.900000, 0.900000, 0.900000], # Week 2
                    [0.100000, 0.439365, 0.535234, 0.100000, 0.367470, 0.975232], # Week 3
                    [0.100000, 0.199729, 0.201684, 0.103729, 0.310943, 0.748939], # Week 4
                    [0.770200, 0.032400, 0.948200, 0.525000, 0.845400, 0.393100], # Week 5
                    [0.958389, 0.843644, 0.711162, 0.307427, 0.256341, 0.429319], # Week 6
                    [0.394426, 0.606437, 0.005884, 0.800916, 0.534595, 0.337790], # Week 7
                    [0.023624, 0.028917, 0.684768, 0.017858, 0.829508, 0.112882], # Week 8
                    [0.100000, 0.198247, 0.173095, 0.082758, 0.313939, 0.763621], # Week 9
                    [0.101717, 0.217655, 0.337649, 0.203732, 0.293678, 0.673045], # Week 10
                    [0.097177, 0.221951, 0.410126, 0.256124, 0.300504, 0.618146], # Week 11
                    [0.083137, 0.214957, 0.413178, 0.260460, 0.309334, 0.607274], # Week 12
                    [0.455074, 0.378790, 0.414111, 0.453655, 0.465969, 0.523820]]), # Week 13
    axis=0
)
# Results (weekly evaluation outputs)
f7_output_data = np.append(f7_output_data, 1.551275860667744) # GP + UCB (iterations = 10) kappa = 8.0
f7_output_data = np.append(f7_output_data, 0.0005215039140009245) # GP + UCB (iterations = 10) kappa = 20.0
f7_output_data = np.append(f7_output_data, 0.9143578861897479) # GP + UCB (iterations = 10) kappa = 4.0 ?????
f7_output_data = np.append(f7_output_data, 1.8757352131149276) # GP + UCB (iterations = 10) kappa = 2.0 ?????
f7_output_data = np.append(f7_output_data, 0.001978776109030435) # Neural Networks + UCB (iterations = 10) kappa 2.0
f7_output_data = np.append(f7_output_data, 0.05806318383973897) # GP + EI (iterations = 10) kappa = 0.05
f7_output_data = np.append(f7_output_data, 0.2926005954433965) # GP + EI (iterations = 15) kappa = 5.0
f7_output_data = np.append(f7_output_data, 0.10481124129157648) # GP + EI (iterations = 15) kappa = 10.0
f7_output_data = np.append(f7_output_data, 1.6525247601700708) # GP + UCB (iterations = 15) kappa = 1.0
f7_output_data = np.append(f7_output_data, 2.8168922468694917) # GP + UCB (iterations = 10) kappa = 0.5
f7_output_data = np.append(f7_output_data, 3.029156813606421) # GP + UCB (iterations = 10) kappa = 0.5
f7_output_data = np.append(f7_output_data, 2.992904995633214) # GP + UCB (iterations = 10) kappa = 0.2
f7_output_data = np.append(f7_output_data, 0.9856429540383217) # GP + UCB + PCA (iterations = 10) kappa = 0.1

In [92]:
# Load the initial data provided
f8_input_data = np.load('./../initial_data/function_8/initial_inputs.npy')
f8_output_data = np.load('./../initial_data/function_8/initial_outputs.npy')

# Weekly submissions
f8_input_data = np.concatenate(
    (f8_input_data, [[0.100000, 0.100000, 0.100000, 0.100000, 0.900000, 0.100000, 0.100000, 0.900000], # Week 1
                     [0.100000, 0.900000, 0.100000, 0.900000, 0.900000, 0.900000, 0.100000, 0.900000], # Week 2
                    [0.100000, 0.058521, 0.118593, 0.021611, 0.900000, 0.748495, 0.157896, 0.576730], # Week 3
                    [0.150061, 0.263393, 0.100000, 0.078784, 0.681402, 0.155068, 0.120381, 0.071284], # Week 4
                    [0.197900, 0.682800, 0.352600, 0.859300, 0.417900, 0.660200, 0.907500, 0.559300], # Week 5
                    [0.297223, 0.725239, 0.450101, 0.260713, 0.017052, 0.457917, 0.171118, 0.506765], # Week 6
                    [0.919172, 0.231566, 0.578186, 0.028455, 0.467396, 0.882039, 0.141356, 0.363101], # Week 7
                    [0.281217, 0.981014, 0.198875, 0.969273, 0.694379, 0.029533, 0.494993, 0.053073], # Week 8
                    [0.044472, 0.358709, 0.000000, 0.256740, 0.542412, 0.569371, 0.102869, 0.580490], # Week 9
                    [0.091430, 0.153888, 0.173259, 0.087793, 0.737778, 0.470758, 0.223349, 0.647339], # Week 10
                    [0.128093, 0.183460, 0.124892, 0.170784, 0.809712, 0.509660, 0.222141, 0.613314], # Week 11
                    [0.134207, 0.173346, 0.133126, 0.150276, 0.757313, 0.500828, 0.208864, 0.572960], # Week 12
                    [0.686136, 0.467986, 0.557699, 0.582667, 0.508812, 0.525244, 0.631114, 0.642329]]), # Week 13
    axis=0
)
# Results (weekly evaluation outputs)
f8_output_data = np.append(f8_output_data, 9.7983) # GP + UCB (iterations = 10) kappa = 8.0
f8_output_data = np.append(f8_output_data, 8.6783) # GP + UCB (iterations = 10) kappa = 20.0
f8_output_data = np.append(f8_output_data, 9.904408090344) # GP + UCB (iterations = 10) kappa = 3.0 ?????
f8_output_data = np.append(f8_output_data, 9.8077148428394) # GP + UCB (iterations = 10) kappa = 2.0 ?????
f8_output_data = np.append(f8_output_data, 7.945254176000001) # Neural Networks + UCB (iterations = 10) kappa 2.0
f8_output_data = np.append(f8_output_data, 8.9608425698375) # GP + EI (iterations = 10) kappa = 0.05
f8_output_data = np.append(f8_output_data, 7.8201193386419) # GP + EI (iterations = 15) kappa = 5.0
f8_output_data = np.append(f8_output_data, 8.127425233881599) # GP + EI (iterations = 15) kappa = 10.0
f8_output_data = np.append(f8_output_data, 9.831285357306) # GP + UCB (iterations = 15) kappa = 2.5
f8_output_data = np.append(f8_output_data, 9.9862489266639) # GP + UCB (iterations = 10) kappa = 0.5
f8_output_data = np.append(f8_output_data, 9.9956530943604) # GP + UCB (iterations = 10) kappa = 0.5
f8_output_data = np.append(f8_output_data, 9.9959433020615) # GP + UCB (iterations = 10) kappa = 0.2
f8_output_data = np.append(f8_output_data, 8.0608638244959) # GP + UCB + PCA (iterations = 10) kappa = 0.1

## Bayesian optimisation: Gaussian Process as surrogate function + UCB and EI as acquistion function

In [93]:
# Acquisition function - UCB
def upper_confidence_bound(mu, sigma, kappa):
    """
    Upper Confidence Bound (UCB) acquisition function.
    
    UCB = mean + kappa * std
    
    Parameters:
    -----------
    mu : predicted mean
    sigma : predicted standard deviation
    kappa : exploration parameter (higher = more exploration)
    """
    return mu + kappa * sigma


In [94]:
def expected_improvement(mu, sigma, y_best, xi=0.01):
    """
    Expected Improvement (EI) acquisition function.
    
    EI = E[max(f(x) - f(x_best), 0)]
    
    Parameters:
    -----------
    mu : predicted mean
    sigma : predicted standard deviation  
    y_best : best observed value so far
    xi : exploration parameter
    """
    with np.errstate(divide='warn'):
        improvement = mu - y_best - xi
        Z = improvement / sigma
        ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0
    return ei

## Note
Ensure you comment/uncomment the acquisition function desired in the bayesian optimization function: `upper_confidence_bound(mu, sigma, kappa=0.1)` and `expected_improvement(mu, sigma, best_values[0], xi=0.2)`. Additionally, set `kappa` or `xi` as desired depending on the exploration-exploitation intend. 

In [79]:
def bayesian_optimization_nd(X_samples, y_samples, n_dims, n_iterations=50, n_initial=10):
    """
    Bayesian Optimization for n-dimensional functions.
    Assumes bounds [0, 1] for all dimensions.
    """
    bounds = [(0, 1) for _ in range(n_dims)]
    
    best_values = [y_samples.max()]

    # Scale outputs
    # y_samples = (y_samples - y_samples.mean()) / y_samples.std()

    X_next_best = []
    
    print(f"Starting {n_dims}D optimization with {n_initial} initial samples...")
    print(f"Initial best y: {y_samples.max():.20f}\n")
    
    for iteration in range(n_iterations):
        # Fit GP
        kernel = ConstantKernel(1.0, (1e-8, 1e3)) * RBF(0.3, (1e-4, 10))
        gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6,
                                     n_restarts_optimizer=5)
        gp.fit(X_samples, y_samples)
        
        # Optimize acquisition function
        def acq_objective(X):
            X = X.reshape(1, -1)
            mu, sigma = gp.predict(X, return_std=True)
            
           # Kappa initially 8.0 , after 20.0, after 12.00 (explore the middle), 2.0 default or used for NeuralNetworks
            acq_val = upper_confidence_bound(mu, sigma, kappa=0.1)

            # Uncomment this line if you would like to apply EI instead of UCB as acquisition function
            # acq_val = expected_improvement(mu, sigma, best_values[0], xi=0.2)
            
            return -acq_val
        
        # Multi-start optimization
        best_acq = np.inf
        X_next = None

        for _ in range(20):  # More starts for higher dimensions
            x0 = np.random.uniform(0, 1, n_dims)
            result = minimize(acq_objective, x0, bounds=bounds, method='L-BFGS-B')
            if result.fun < best_acq:
                best_acq = result.fun
                X_next = result.x
                X_next_best.append(X_next)
                print("Best Next X is: " + str(X_next))

    print(len(X_next_best))
    return X_next_best, y_samples, best_values


## Apply Gaussian Processes with UCB/EI

Replace the function input `f8_input_data`, output `f8_output_data` and `n_dims` dimensions variables appropriately.

In [96]:
# Run the D-dimensional function
X_samples_generated, y_samples_generated, best_values_generated = bayesian_optimization_nd(
    f8_input_data, f8_output_data, 
    n_dims=8,
    n_iterations=10,
)

Starting 8D optimization with 10 initial samples...
Initial best y: 9.99594330206149983553

Best Next X is: [0.10869386 0.18649289 0.14586067 0.15310158 0.77359643 0.53229195
 0.22340099 0.68454218]
Best Next X is: [0.10869012 0.18649656 0.14586756 0.15310692 0.77360798 0.53232741
 0.22341964 0.68463908]
Best Next X is: [0.10869178 0.18649196 0.1458671  0.15310068 0.77362419 0.53234623
 0.22342161 0.6846468 ]
Best Next X is: [0.10870231 0.18650836 0.14587519 0.15310769 0.7736315  0.53237913
 0.22342865 0.68468262]
Best Next X is: [0.1086819  0.18648326 0.1458691  0.15309033 0.77363283 0.53233653
 0.22341255 0.68471945]
Best Next X is: [0.10869902 0.18650169 0.14586698 0.15309231 0.77360842 0.53235121
 0.22340797 0.68460642]
Best Next X is: [0.10868979 0.18649986 0.14586915 0.15308536 0.77362415 0.53234251
 0.22341333 0.68468158]
Best Next X is: [0.10869182 0.18649725 0.14586687 0.15309627 0.7736151  0.53232578
 0.22341202 0.68464358]
Best Next X is: [0.10868718 0.18649111 0.14586748 0.

# Gaussian Process with PCA for dimensionality reduction

In [77]:
def apply_pca(X, y):
    """
    Apply PCA.
    
    Parameters:
    -----------
    X : input value dimensions
    y : output values for the dimensions
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=2)  # reduced from number of dimensions to 2-dimensional
    X_reduced = pca.fit_transform(X_scaled)

    X_samples, y_samples, best_values_8d = bayesian_optimization_nd(X_reduced, y, n_dims=2, n_iterations=10)
    for X_sample in X_samples:
        X_reshaped = X_sample.reshape(1, -1)
        # Step 1: inverse PCA
        x_next_scaled = pca.inverse_transform(X_reshaped)

        # Step 2: inverse scaling
        x_next_original = scaler.inverse_transform(x_next_scaled)
        print("Bext Next X after inverse is : " + str(x_next_original))
    
    return X_reduced


# Apply Gaussian Process with PCA for dimensionality reduction

In [78]:
# Apply PCA on high dimensional function with GP as surrogate function and UCB as acquistion function
# Update the input values here corresponding with the target function number 
apply_pca(f8_input_data, f8_output_data)

Starting 2D optimization with 10 initial samples...
Initial best y: 9.99594330206149983553

Best Next X is: [0.32719834 0.92129442]
Best Next X is: [0.3271981 0.9212946]
Best Next X is: [0.54068603 0.        ]
Best Next X is: [0.54068602 0.        ]
Best Next X is: [0.32719818 0.92129453]
Best Next X is: [0.54068601 0.        ]
Best Next X is: [0.54068602 0.        ]
Best Next X is: [0.32719804 0.92129468]
Best Next X is: [0.32719797 0.92129457]
Best Next X is: [0.32719803 0.92129462]
Best Next X is: [0.54068603 0.        ]
Best Next X is: [0.24362535 0.20827244]
Best Next X is: [0.32719826 0.92129445]
Best Next X is: [0.54068598 0.        ]
Best Next X is: [0.540686 0.      ]
Best Next X is: [0.54068611 0.        ]
Best Next X is: [0.32719898 0.92129454]
Best Next X is: [0.32719866 0.92129432]
Best Next X is: [0.54068587 0.        ]
Best Next X is: [0.540686 0.      ]
Best Next X is: [0.3271934  0.92129702]
Best Next X is: [0.32719363 0.92129693]
Best Next X is: [0.54068621 0.        

array([[ 0.59716955,  0.53431186],
       [ 0.98163755, -1.5493474 ],
       [ 0.14566047, -2.2940984 ],
       [ 0.96288271, -1.6089314 ],
       [ 0.49876097,  0.17298725],
       [-0.45444913,  1.6080779 ],
       [ 0.46437991,  2.3428119 ],
       [ 1.52267565, -1.26337014],
       [ 1.12671081, -1.33255984],
       [ 1.97844979, -0.80428454],
       [ 0.72227362,  0.16589521],
       [ 1.83874567, -0.1037277 ],
       [-0.34044636, -2.23207717],
       [ 1.16682988,  1.53388385],
       [-2.52383583,  0.67936919],
       [ 0.34582402,  0.73863946],
       [ 1.61269813, -1.43969178],
       [ 1.20155098, -0.13801655],
       [ 0.82236256, -0.82587126],
       [-0.28530869, -1.6329213 ],
       [ 1.34855646,  1.84737957],
       [ 1.67865449,  0.212194  ],
       [-0.92800654,  0.69690774],
       [-0.28614162, -0.19473426],
       [ 0.93874045,  0.46122616],
       [-0.65981719, -0.16603153],
       [-0.75543358, -0.48178089],
       [ 0.5132795 ,  2.37752255],
       [-0.07447379,

# Neural Networks as surrogate function with UCB as acquisition function

In [73]:
# Neural Networks using torch with gradient descent 
# PyTorch core
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class SurrogateFunctionNeuralNetworks(nn.Module):
    def __init__(self, input_dimensions):
        super().__init__()
        # Define layers
        self.fc1 = nn.Linear(input_dimensions, 64)  # input -> hidden 1
        self.fc2 = nn.Linear(64, 32)          # hidden 1 -> hidden 2
        self.fc3 = nn.Linear(32, 1)           # hidden 2 -> output
        self.dropout = nn.Dropout(p=0.3)      # regularisation

    def forward(self, x):
        x = F.relu(self.fc1(x))   # first hidden layer
        x = self.dropout(x)       # drop 30 % of neurons during training
        x = F.relu(self.fc2(x))   # second hidden layer
        x = self.dropout(x)
        x = torch.sigmoid(self.fc3(x))  # output probability in [0, 1]
        return x




In [76]:
# Epoch looop and get the suggested inputs
def predict(model, x, n_samples=5):
    # model.train()  # important: keep dropout active
    preds = torch.stack([model(x) for _ in range(n_samples)])
    mean = preds.mean(dim=0)
    std = preds.std(dim=0)
    return mean, std

def get_next_best_points(inputs, outputs, dimensions, iterations):
    X_training = torch.tensor(inputs, dtype=torch.float32)
    y_training = torch.tensor(outputs, dtype=torch.float32)

    model  = SurrogateFunctionNeuralNetworks(dimensions)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    bounds = [(0, 1) for _ in range(dimensions)]
    
    best_values = [y_training.max()]

    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        pred = model(X_training)
        loss = F.mse_loss(pred.squeeze(), y_training)
        loss.backward()
        optimizer.step()

    # Neural Network kappa = 2.0 for all functions
    def acq_objective(x):
        # x = torch.rand(1, dimensions, requires_grad=True)
        # print(x)
        mu, sigma = predict(model, x)
            
        acq_val = upper_confidence_bound(mu, sigma, kappa=2.0)
            
        return -acq_val

    x = torch.rand(1, dimensions, requires_grad=True)
    optimizer = torch.optim.Adam([x], lr=1e-3)

    for _ in range(20):
        optimizer.zero_grad()
        ei = acq_objective(x)
        loss = -ei  # maximize EI
        loss.backward()
        optimizer.step()
        x.data.clamp_(0.0, 1.0)  # enforce bounds

    # print(x.detach())
    return x.detach()
        
    # Multi-start optimization
    # best_acq = np.inf
    # X_next = None
        
    # for _ in range(20):  # More starts for higher dimensions
    #     x0 = np.random.uniform(0, 1, dimensions)
    #     print(x0)
    #     result = minimize(acq_objective, x0, bounds=bounds, method='L-BFGS-B')
    #     result = result.detach().numpy()
    #     print(result)
    #     if result.fun < best_acq:
    #         best_acq = result.fun
    #         X_next = result.x
    #         print("Best Next X is: " + str(X_next))
    
    # return  best_values


# Update the input variables with the target function number 
_ = get_next_best_points(
    f1_input_data, f1_output_data, 
    dimensions=2,
    iterations=10,
)

# Update the input variables with the target function number 
for iteration in range(20):
    x_next = get_next_best_points(f8_input_data, f8_output_data, dimensions=8,iterations=10)
    print(x_next)

    

tensor([[0.7146, 0.0668, 0.5920, 0.5650, 0.9806, 0.7212, 0.2372, 0.5498]])
tensor([[0.9427, 0.9071, 0.5252, 0.8269, 0.9487, 0.4751, 0.8228, 0.1718]])
tensor([[0.6075, 0.4647, 0.5919, 0.7636, 0.8765, 0.5795, 0.3819, 0.2653]])
tensor([[0.6263, 0.9455, 0.5887, 0.9107, 0.6360, 0.5754, 0.7104, 0.2343]])
tensor([[0.9964, 0.8114, 0.1974, 0.9940, 0.6287, 0.4106, 0.9371, 0.1726]])
tensor([[0.6291, 0.9791, 0.3949, 0.3291, 0.8617, 0.7258, 0.5268, 0.0983]])
tensor([[0.9679, 0.6017, 0.0535, 0.4634, 0.2178, 0.6437, 0.2615, 0.9090]])
tensor([[0.2994, 0.6039, 0.5568, 0.3201, 0.8472, 0.4107, 0.1577, 0.9967]])
tensor([[0.8029, 0.3289, 0.3296, 0.0505, 0.8474, 0.5364, 0.0985, 0.2496]])
tensor([[0.2123, 0.3868, 0.7880, 0.3795, 0.3175, 0.8977, 0.1755, 0.9091]])
tensor([[0.2894, 0.5120, 0.2370, 0.7184, 0.0872, 0.3139, 1.0000, 1.0000]])
tensor([[0.2439, 0.8907, 0.0177, 0.1892, 0.9456, 0.1444, 0.8438, 0.8867]])
tensor([[0.0761, 0.1684, 0.0477, 0.9817, 0.0920, 0.5461, 0.1433, 0.8283]])
tensor([[0.5350, 0.5846, 